In [17]:
from analysis.processors.base import BaseProcessor
from coffea.nanoevents import NanoAODSchema
from coffea import processor
import uproot
import warnings
NanoAODSchema.warn_missing_crossrefs = False
!voms-proxy-init --voms cms > /dev/null 2>&1

workflow = "btag_eff_ele"
year = "2022preEE"

fileset = {'2022preEE': {
#     'TTTo2L2Nu_1': {
#         'files': {'root://cms-xrd-global.cern.ch:1094//store/mc/Run3Summer22NanoAODv12/TTto2L2Nu_TuneCP5_13p6TeV_powheg-pythia8/NANOAODSIM/130X_mcRun3_2022_realistic_v5-v2/50000/d51aa7d0-59ab-4ce7-8852-3f9a1ec14bc6.root': 'Events'},
#         'metadata': {'short_name': 'TTTo2L2Nu'}},
#    'SingleMuonC_1': {
#       'files': {"root://cms-xrd-global.cern.ch:1094//store/data/Run2022C/DoubleMuon/NANOAOD/22Sep2023-v1/50000/9f317280-5380-49f7-8a30-3e3bcd48dc9c.root": "Events"},
#       'metadata': {'short_name': 'SingleMuon'}},
    'WJetsToLNu_0to120_HT100to400_1': {
        'files': {"root://cms-xrd-global.cern.ch:1094//store/mc/Run3Summer22NanoAODv12/WtoLNu-4Jets_MLNu-120_HT-1500to2500_TuneCP5_13p6TeV_madgraphMLM-pythia8/NANOAODSIM/130X_mcRun3_2022_realistic_v5-v2/40000/a008699b-a67b-4f2e-8712-7133198d0770.root": "Events"},
        'metadata': {'short_name': 'WJetsToLNu'}
    }
}
          }

In [20]:
futures_run = processor.Runner(
    executor=processor.FuturesExecutor(workers=1, compression=None, retries=3),
    schema=NanoAODSchema,
    chunksize=50000,
    savemetrics=False,
    xrootdtimeout=120,
    align_clusters=True
)
out = futures_run(
    fileset[year], 
    treename="Events", 
    processor_instance=BaseProcessor(workflow=workflow, year=year, mode="virtual"),
)
out["metadata"]

Output()

There are 257 events with muon pt outside of [26,200] GeV. Setting those entries to their initial value.
There are 257 events with muon pt outside of [26,200] GeV. Setting those entries to their initial value.
There are 3 nan entries in the corrected pt. This might be due to the number of tracker layers hitting boundaries. Setting those entries to their initial value.


{'sumw': np.float32(24.903229),
 'base': {'cutflow': {'initial': np.float32(24.903229),
   'goodvertex': np.float32(24.903229),
   'lumi': np.float32(24.903229),
   'trigger': np.float32(7.1492524),
   'trigger_match': np.float32(7.0300975),
   'metfilters': np.float32(6.9705205),
   'hemcleaning': np.float32(6.9705205),
   'met_50': np.float32(6.3747497),
   'tau_veto': np.float32(6.3747497),
   'muon_veto': np.float32(6.3747497),
   'exactly_one_electron': np.float32(4.766168)},
  'weighted_final_nevents': np.float64(4.272125484609193)}}

In [ ]:
from coffea.nanoevents import NanoEventsFactory, NanoAODSchema
filename = list(fileset[year]['WJetsToLNu_0to120_HT100to400_1']['files'].keys())[0]
events = NanoEventsFactory.from_root(
    filename,
    treepath="Events",
    entry_stop=1_000,
    metadata={"dataset": "WJetsToLNu_0to120_HT100to400_1"},
    schemaclass=NanoAODSchema,
    mode="virtual",
).events()

OSError: Failed to open file: [ERROR] Server responded with an error: [3011] No servers are available to read the file.


In [ ]:
filename

'root://cms-xrd-global.cern.ch//store/mc/Run3Summer22EENanoAODv12/WtoLNu-4Jets_MLNu-0to120_HT-100to400_TuneCP5_13p6TeV_madgraphMLM-pythia8/NANOAODSIM/130X_mcRun3_2022_realistic_postEE_v6-v3/50000/27e1554b-3d69-4277-8549-88a697014d63.rootroot://cms-xrd-global.cern.ch//store/mc/Run3Summer23NanoAODv12/WtoLNu-4Jets_MLNu-0to120_HT-100to400_TuneCP5_13p6TeV_madgraphMLM-pythia8/NANOAODSIM/130X_mcRun3_2023_realistic_v14-v2/30000/0796aa97-7c9c-4b29-ac70-4f95856f7608.root'

### Why HTCondor dask cluster doesn't work in SWAN?

In [ ]:
from dask.distributed import Client
client = Client("tls://10.100.197.122:30795")
dask_run = processor.Runner( #this is new runner function for coffea 2025.10
    executor=processor.DaskExecutor(client=client, compression=None), #execute via dask workers
    schema=NanoAODSchema,
    chunksize=100_000,
    skipbadfiles=False,
    savemetrics=False,
)
#histograms = dask_run(test_fileset, processor_instance=MuonProcessor(workflow_path, year))
histograms = dask_run(fileset[year], treename="Events", processor_instance=BaseProcessor(workflow=workflow, year=year, mode="virtual"))

In [14]:
from coffea.dataset_tools import apply_to_fileset, max_chunks, max_files, preprocess
preprocessed_available, preprocessed_total = preprocess(
    fileset[year],
    step_size=100_000,
    align_clusters=False,
    skip_bad_files=False,
    recalculate_steps=False,
    files_per_batch=1,
    file_exceptions=(OSError,),
    save_form=False,
    uproot_options={},
    step_size_safety_factor=0.5,
)

In [ ]:
test_preprocessed_files = max_files(preprocessed_available, 1)
test_preprocessed = max_chunks(test_preprocessed_files, 3)
small_tg, small_rep = apply_to_fileset(
    data_manipulation=BaseProcessor(workflow=workflow, year=year, mode="dask"),
    fileset=test_preprocessed,
    schemaclass=NanoAODSchema,
    uproot_options={"allow_read_errors_with_report": (OSError, ValueError)},
)